# Configurable Single-Experiment Vibration Generator

This notebook creates exactly one synthetic vibration experiment. It supports `single_sine`, `sparse_multi_sine`, `dense_multi_sine`, `band_limited_random_phase`, `chirp`, and `mixed_bands`. The notebook contains no scientific implementation: it calls the same `generate_experiment` engine used by the Python script.

Run the cells from top to bottom. Edit only the parameter cell unless you are developing the package. V2E is optional at runtime: when it is unavailable, all base artifacts are retained and `events.h5` is intentionally absent.

In [ ]:
from pathlib import Path
import json
import os
import sys

candidate = Path.cwd().resolve()
if (candidate / 'event_vibration_simulator').is_dir():
    PROJECT_ROOT = candidate
elif (candidate.parent / 'event_vibration_simulator').is_dir():
    PROJECT_ROOT = candidate.parent
else:
    raise RuntimeError('Run this notebook from the package root or notebooks directory.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from event_vibration_simulator import deep_merge, default_parameters, generate_experiment
from event_vibration_simulator.config import resolve_parameters

print(f'Package root: {PROJECT_ROOT}')

## User Parameters

All configurable values are collected here. Only the section matching `motion.type` is active, but the other scenario defaults remain visible for easy switching. `EVENT_VIBRATION_CONFIG_JSON` may provide a recursive JSON override for automated notebook execution.

In [ ]:
USER_PARAMETERS = default_parameters()

USER_PARAMETERS['experiment'].update({
    'name': 'notebook_single_sine_500hz_default',
    'output_root': 'generated_experiments',
    'output_policy': 'new',              # new | error | overwrite
    'random_seed': 7301,
    'write_manifest': True,
})

USER_PARAMETERS['sampling'].update({
    'video_fps': 40000.0,
    'duration_mode': 'seconds',          # seconds | cycles
    'duration_sec': 0.2,
    'number_of_cycles': 40.0,
    'cycle_reference_frequency_hz': 500.0,
    'width_px': 240,
    'height_px': 180,
    'save_frames': True,
    'recommended_samples_per_cycle': 20.0,
})

USER_PARAMETERS['rendering'].update({
    'object_shape': 'circle',            # circle | rectangle | square
    'marker_radius_px': 5.0,
    'rectangle_width_px': 10.0,
    'rectangle_height_px': 10.0,
    'base_position_x_px': 120.0,
    'base_position_y_px': 90.0,
    'background_intensity': 5,
    'marker_contrast': 0.72,
    'video_codec': 'libx264',
    'video_crf': 28,
    'video_preset': 'veryfast',
})

USER_PARAMETERS['calibration'].update({
    'meters_per_pixel': None,            # Set only when physically known
    'standard_gravity_m_s2': 9.80665,
})

USER_PARAMETERS['motion'].update({
    # single_sine | sparse_multi_sine | dense_multi_sine |
    # band_limited_random_phase | chirp | mixed_bands
    'type': 'single_sine',
    'frequency_min_hz': 50.0,
    'frequency_max_hz': 2000.0,
    'normalization_mode': 'peak_displacement_px',
    'normalization_value': 6.0,
    'two_axis_motion': False,
    'y_amplitude_ratio': 0.25,
    'custom_frequencies_hz': [],
    'custom_weights': [],
    'custom_phases_x_rad': [],
    'custom_phases_y_rad': [],
})
USER_PARAMETERS['motion']['single_sine'].update({
    'frequency_hz': 500.0,
    'phase_x_rad': 0.0,
    'phase_y_rad': 1.5707963267948966,
})
USER_PARAMETERS['motion']['sparse_multi_sine'].update({
    'num_components': 8,
    'minimum_frequency_separation_hz': 35.0,
    'weight_min': 0.6,
    'weight_max': 1.3,
})
USER_PARAMETERS['motion']['dense_multi_sine'].update({
    'num_components': 32,
    'weight_lognormal_mean': 0.0,
    'weight_lognormal_sigma': 0.45,
})
USER_PARAMETERS['motion']['band_limited_random_phase'].update({
    'num_components': 80,
    'spectral_variant': 'flat',          # flat | low_tilt | mid_bump
    'mid_bump_center_hz': 800.0,
    'mid_bump_sigma_hz': 450.0,
})
USER_PARAMETERS['motion']['chirp'].update({
    'start_frequency_hz': 50.0,
    'end_frequency_hz': 2000.0,
    'kind': 'linear',                    # linear | log
    'phase_x_rad': 0.0,
    'phase_y_rad': 1.5707963267948966,
})
USER_PARAMETERS['motion']['mixed_bands'].update({
    'bands_hz': [[50.0, 200.0], [1000.0, 2000.0]],
    'components_per_band': [6, 6],
    'weight_ranges': [[0.8, 1.2], [0.6, 1.0]],
})

USER_PARAMETERS['v2e'].update({
    'enabled': True,
    'command': '',                       # Optional v2e executable
    'script_path': '',                   # Optional full path to v2e.py
    'python_executable': '',             # Optional V2E-environment Python
    'disable_slomo': True,
    'pos_thres': 0.2,
    'neg_thres': 0.2,
    'sigma_thres': 0.03,
    'refractory_period_sec': 0.0001,
    'shot_noise_rate_hz': 0.0,
    'cutoff_hz': 0.0,
    'dvs_emulator_seed': None,           # None uses experiment.random_seed
    'timeout_sec': 0,                    # 0 means unlimited
})

automation_override = os.environ.get('EVENT_VIBRATION_CONFIG_JSON')
if automation_override:
    USER_PARAMETERS = deep_merge(USER_PARAMETERS, json.loads(automation_override))

USER_PARAMETERS

## Validate

This calculates frame count, duration, maximum active frequency, samples per cycle, and the resolved V2E seed without creating experiment files.

In [ ]:
resolved_parameters, validation_warnings = resolve_parameters(USER_PARAMETERS, PROJECT_ROOT)
print(json.dumps({
    'experiment': resolved_parameters['experiment']['name'],
    'motion_type': resolved_parameters['motion']['type'],
    'duration_sec': resolved_parameters['derived']['duration_sec'],
    'frame_count': resolved_parameters['derived']['frame_count'],
    'maximum_active_frequency_hz': resolved_parameters['derived']['maximum_active_frequency_hz'],
    'samples_per_highest_frequency_cycle': resolved_parameters['derived']['samples_per_highest_frequency_cycle'],
    'resolved_v2e_seed': resolved_parameters['v2e']['resolved_dvs_emulator_seed'],
    'warnings': validation_warnings,
}, indent=2))

## Generate the Single Experiment

The call below is the same API used by `generate_single_experiment.py`. Default generation writes 8,000 PNG files and runs V2E when it is available, so it can take time.

In [ ]:
result = generate_experiment(USER_PARAMETERS, project_root=PROJECT_ROOT)
print(json.dumps(result, indent=2))

## Inspect the Generated Artifacts

In [ ]:
from IPython.display import Image, display
import pandas as pd

experiment_dir = Path(result['experiment_directory'])
display(pd.read_csv(experiment_dir / 'experiment_summary.csv'))
display(Image(filename=str(experiment_dir / 'preview.png')))
display(Image(filename=str(experiment_dir / 'plots' / 'ground_truth_psd.png')))

In [ ]:
import h5py
from event_vibration_simulator.v2e_runner import count_events

events_path = experiment_dir / 'events.h5'
if events_path.exists():
    with h5py.File(events_path, 'r') as h5:
        print('HDF5 keys:', list(h5.keys()))
    print('Event count:', count_events(events_path))
else:
    status = json.loads((experiment_dir / 'event_conversion_status.json').read_text())
    print('events.h5 is absent by design.')
    print(json.dumps(status, indent=2))